<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase7-rag-observer-agent/02_observer_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7: Observer Agent Architecture
**Goal**: Build a second agent that audits RAG pipeline
      outputs for grounding quality, source relevance,
      and factual integrity before answers reach users.
**Regulatory mapping**: NIST AI 600-1 output monitoring,
                    EU AI Act Article 14 human oversight.
**Date**: June 2026.
**Status**: In Progress

In [ ]:
import time
import json
import pandas as pd
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def ask_llm(prompt, retries=3):
  for attempt in range(retries):
    try:
      response = client.models.generate_content(
          model="gemini-flash-latest",
          contents=prompt
      )
      return response.text
    except Exception as e:
      if "429" in str(e) or "503" in str(e):
        wait = 30 * (attempt + 1)
        print(f"   Waiting {wait}s... (attempt {attempt + 1}/{retries})")
        time.sleep(wait)
      else:
        raise e
  raise Exception("LLM call failed after multiple retries")

# - KNOWLEDGE BASE (same as I used before) -
KNOWLEDGE_BASE = {
    "doc_001": {
        "title":    "EU AI Act Article 5 - Prohibited Practices",
        "content":  """Article 5 of the EU AI Act prohibits the following
AI practices: subliminal manipulation below the threshold of consciousness,
exploitation of vulnerabilities of specific groups, social scoring by public
authorities, real-time remote biometric identification in public spaces by
law enforcement except in narrowly defined cases, retrospective biometric
identification systems, emotion recognition in workplace and educational
settings, and biometric categorisation to infer sensitive characteristics
such as race, political opinions, or sexual orientation.""",
        "source":   "EU AI Act 2024/1689, Article 5",
        "category": "regulation"
    },
    "doc_002": {
        "title":    "EU AI Act Article 10 - Data Governance",
        "content":  """Article 10 requires that training, validation and
testing datasets for high-risk AI systems must be relevant, representative,
free of errors and complete. They must be examined for possible biases that
could lead to risks to fundamental rights or discrimination. When processing
special categories of personal data, appropriate safeguards must be in place.
Providers must implement data governance practices covering the intended
purpose, data collection processes, and examination for biases.""",
        "source":   "EU AI Act 2024/1689, Article 10",
        "category": "regulation"
    },
    "doc_003": {
        "title":    "EU AI Act Article 14 - Human Oversight",
        "content":  """Article 14 requires that high-risk AI systems be
designed to allow effective oversight by natural persons during the period
of use. The system must enable those responsible for oversight to understand
the capabilities and limitations of the system, detect and address
malfunctions, and intervene or interrupt the system through a stop button
or similar procedure. Human oversight must be proportionate to the risks.""",
        "source":   "EU AI Act 2024/1689, Article 14",
        "category": "regulation"
    },
    "doc_004": {
        "title":    "EU AI Act Article 99 - Penalties",
        "content":  """Article 99 establishes three penalty tiers.
Violations of prohibited practices under Article 5 carry fines of up to
35 million euros or 7 percent of global annual turnover. Violations of
obligations for high-risk AI systems under Chapters III and V carry fines
of up to 15 million euros or 3 percent of global annual turnover under
Article 99(3). Providing incorrect or misleading information to authorities
carries fines of up to 7.5 million euros or 1 percent of global annual
turnover.""",
        "source":   "EU AI Act 2024/1689, Article 99",
        "category": "regulation"
    },
    "doc_005": {
        "title":    "NIST AI RMF - Four Core Functions",
        "content":  """The NIST AI Risk Management Framework organises
risk management into four core functions. GOVERN establishes organisational
policies, culture, and accountability for AI risk. MAP identifies the
context and specific risks of each AI application. MEASURE uses
quantitative and qualitative methods to analyse and assess AI risks.
MANAGE prioritises and acts on identified risks through controls,
monitoring, and incident response. These functions are intended to be
applied continuously across the AI lifecycle.""",
        "source":   "NIST AI RMF 1.0, Core Functions",
        "category": "framework"
    },
    "doc_006": {
        "title":    "NIST AI 600-1 — Generative AI Risk",
        "content":  """NIST AI 600-1 addresses risks specific to generative
AI systems including large language models. Key risks include hallucination
where models produce plausible but false information, data memorisation
where models reproduce training data, harmful bias in generated content,
and confabulation of sources and citations. Organisations are required to
implement hallucination tracking, source data lineage verification, and
output monitoring to address these risks in production deployments.""",
        "source":   "NIST AI 600-1, Generative AI Profile",
        "category": "framework"
    },
}

# - RETRIEVAL ENGINE (same as I used previously in the last work) -
def retrieve_documents(query, top_k=2):
  """Retrieve most relevant documents with domain boosts."""
  query_lower = query.lower()
  scores      = []

  # Domain-specific boost keywords
  penalty_keywords = ["fine", "penalty", "penalties",
                      "sanction", "euros", "turnover", "amount"]
  oversight_keywords = ["oversight", "human", "intervention",
                        "stop", "interrupt", "monitor"]

  for doc_id, doc in KNOWLEDGE_BASE.items():
    score = 0
    content_lower = (doc["title"] + " " +
                     doc["content"]).lower()
    query_words   = query_lower.split()

    for word in query_words:
      if len(word) > 3:
        if word in content_lower:
          score += 1

    if any(word in doc['title'].lower()
      for word in query_words if len(word) > 3):
      score += 3

    # Apply domain-specific boosts
    if any(k in query_lower for k in penalty_keywords):
      if "article 99" in doc["source"].lower():
        score += 5
      if "article 5" in doc["source"].lower():
        score -= 2

    if any(k in query_lower for k in oversight_keywords):
      if "article 14" in doc["source"].lower():
        score += 5

    scores.append((doc_id, score))

  scores.sort(key=lambda x: x[1], reverse=True)
  return [KNOWLEDGE_BASE[doc_id]
          for doc_id, score in scores[:top_k]
          if score > 0]

def rag_query(query, top_k=2):
  """RAG pipeline with source traceability."""
  retrieved_docs = retrieve_documents(query, top_k)

  if not retrieved_docs:
    return {"query": query, "answer": "No relevant documents found.",
            "source": [], "grounded": False, "doc_count": 0,
            "titles": [], "contents": []}

  context = "\n\n".join([
      f"SOURCE: {doc['source']}\n{doc['content']}"
      for doc in retrieved_docs
  ])

  grounded_prompt = f"""You are an AI governance assistant.
Answer the question using ONLY the provided source documents.
Do not use any knowledge outside of these documents.
If the documents do not contain the answer, say so explicitly.

SOURCE DOCUMENTS:
{context}

QUESTION: {query}

ANSWER (based only on the source documents above):"""

  try:
    answer = ask_llm(grounded_prompt)
  except Exception as e:
    answer = f"Error from LLM: {e}"
    return {"query": query,
            "answer": answer,
            "source": [doc["source"] for doc in retrieved_docs],
            "titles": [doc["title"] for doc in retrieved_docs],
            "contents": [doc["content"] for doc in retrieved_docs],
            "grounded": False,
            "doc_count": len(retrieved_docs)
      }

  return {"query": query,
          "answer": answer,
          "source": [doc["source"] for doc in retrieved_docs],
          "titles": [doc["title"] for doc in retrieved_docs],
          "contents": [doc["content"] for doc in retrieved_docs],
          "grounded": True,
          "doc_count": len(retrieved_docs)
    }

print("====== OBSERVER AGENT SETUP COMPLETE ======")
print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print(f"Retrieval engine: keyword scoring with domain boosts")
print("\nRAG pipeline ready ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== OBSERVER AGENT SETUP COMPLETE ======
Knowledge base: 6 documents
Retrieval engine: keyword scoring with domain boosts

RAG pipeline ready ✅


In [ ]:
# - OBSERVER AGENT -
# A second independent LLM call that audits the
# RAG pipeline output before it reaches the user

def observer_agent(query, rag_result):
  """
  Audits a RAG response for grounding quality,
  source relevance, and factual integrity.
  Returns a structured audit verdict.
  """
  # Handle cases where RAG generation itself failed
  if not rag_result["grounded"] and "Error from LLM" in rag_result["answer"]:
    return {
       "grounding_source":   0,
       "source_relevance":  0,
       "factual_integrity": 0,
       "issues":           [f"RAG LLM call failed: {rag_result['answer']}"],
       "verdict":           "REJECTED"
    }

  # Build the observer prompt
  sources_text = "\n".join([
      f"source {i+1}: {src}"
      for i, src in enumerate(rag_result["source"])
  ])

  # Define content_text using the titles and contents from rag_result
  content_text = "\n\n".join([
      f"TITLE: {title}\nCONTENT: {content}"
      for title, content in zip(rag_result["titles"], rag_result["contents"])
  ])

  observer_prompt = f"""You are an independent AI governance auditor.
Your job is to evaluate whether an AI-generated answer is properly
grounded in its source documents.

ORIGINAL QUESTION:
{query}

SOURCE DOCUMENTS RETRIEVED:
{content_text}

AI-GENERATED ANSWER TO AUDIT:
{rag_result["answer"]}

Evaluate the answer on these three criteria and respond
in JSON format only, no markdown, no extra text:

1. grounding_source: Integer 1-5. How well does the answer
   stay within the source documents?
   5 = entirely grounded, 1 = mostly hallucinated

2. source_relevance: Integer 1-5. How relevant are the
   retrieved sources to the original question?
   5 = entirely relevant, 1 = mostly irrelevant

3. factual_integrity: Integer 1-5. Does the answer
   accurately reflect what the source say?
   5 = entirely factual, 1 = mostly inaccurate

4. issues: List of specific problems found, empty list
   if none.

5. verdict: One of APPROVED, FLAGGED, or REJECTED
   APPROVED = grounding >= 4, relevance >= 4, integrity >= 4
   FLAGGED = any score is 3
   REJECTED = any score is 1 or 2

Return exactly this JSON format:
{{
  "grounding_source": <1-5>,
  "source_relevance": <1-5>,
  "factual_integrity": <1-5>,
  "issues": ["issue 1", "issue 2"],
  "verdict": "APPROVED or FLAGGED or REJECTED"
}}"""

  try:
    raw_result = ask_llm(observer_prompt)
  except Exception as e:
    print(f"    Observer LLM call failed: {e}")
    return {
       "grounding_source":   0,
       "source_relevance":  0,
       "factual_integrity": 0,
       "issues":           [f"Observer LLM call failed: {e}"],
       "verdict":           "REJECTED"
    }

  # Parse JSON response
  import re
  try:
      clean = re.sub(r"'''json/'''","", raw_result).strip()
      match = re.search(r'\{.*\}', clean, re.DOTALL)
      if match:
        scores = json.loads(match.group())
        return scores
  except Exception as e:
      print(f"    Observer parse error: {e}")

  return {
     "grounding_score":   0,
     "source_relevance":  0,
     "factual_integrity": 0,
     "issues":           ["Observer could not parse result"],
     "verdict":           "FLAGGED"
  }

def full_pipeline(query):
  """
  Complete pipeline: RAG generation + Observer audit.
  Returns both the answer and the audit verdict.
  """
  print(f"\nQuery: {query}")
  print("-" * 55)

  # Step 1: RAG generation
  print("Step 1: RAG Agent generating answer...")
  rag_result = rag_query(query)
  print(f"    Sources: {rag_result['source']}")
  print(f"        Answer preview: {rag_result['answer'][:300]}...")
  time.sleep(15)

  # Step 2: Observer audit
  print("Step 2: Observer Agent auditing answer...")
  audit = observer_agent(query, rag_result)
  time.sleep(15)

  # Step 3: Final verdict
  verdict   = audit.get("verdict", "FLAGGED")
  grounding = audit.get("grounding_source", 0)
  relevance = audit.get("source_relevance", 0)
  integrity = audit.get("factual_integrity", 0)
  issues    = audit.get("issues", [])

  print(f"\n    Grounding score: {grounding}/5")
  print(f"    Source Relevance:  {relevance}/5")
  print(f"    Factual Integrity: {integrity}/5")
  print(f"    Issues:            {issues}")
  print(f"    VERDICT:           {verdict}")

  return {
      "query":               query,
      "answer":              rag_result["answer"],
      "source":              rag_result["source"],
      "grounding_score":     grounding,
      "source_relevance":    relevance,
      "factual_integrity":   integrity,
      "issues":              json.dumps(issues),
      "verdict":             verdict,
  }

# - RUN THE FULL PIPELINE -
print("====== OBSERVER AGENT PIPELINE ======")

test_queries = [
    "What does Article 14 require for human oversight?",
    "What are the penalty tiers in the EU AI Act?",
    "What does NIS AI 600-1 say about hallucination?",
]

pipeline_results = []

for query in test_queries:
  result = full_pipeline(query)
  pipeline_results.append(result)

df_observer = pd.DataFrame(pipeline_results)

print("\n====== OBSERVER AUDIT SUMMARY ======")
print(f"Total queries audited:    {len(df_observer)}")
print(f"Total verdict APPROVED:    {len(df_observer[df_observer['verdict'] == 'APPROVED'])}")
print(f"Total verdict FLAGGED:    {len(df_observer[df_observer['verdict'] == 'FLAGGED'])}")
print(f"Total verdict REJECTED:   {len(df_observer[df_observer['verdict'] == 'REJECTED'])}")
print(f"Avg grounding score:      {df_observer['grounding_score'].mean():.1f}/5")
print(f"Avg source relevance:     {df_observer['source_relevance'].mean():.1f}/5")
print(f"Avg factual integrity:    {df_observer['factual_integrity'].mean():.1f}/5")

df_observer.to_csv(SAVE_PATH + "observer_agent_results.csv", index=False)
print(f"Results saved to {SAVE_PATH}observer_agent_results.csv✅")

df_observer[["query", "source", "grounding_score",
             "source_relevance", "verdict"]]

====== OBSERVER AGENT PIPELINE ======

Query: What does Article 14 require for human oversight?
-------------------------------------------------------
Step 1: RAG Agent generating answer...
    Sources: ['EU AI Act 2024/1689, Article 14', 'EU AI Act 2024/1689, Article 10']
        Answer preview: Based on the provided source documents, Article 14 requires that high-risk AI systems be designed to allow effective oversight by natural persons during the period of use. Specifically, it requires that the system enables those responsible for oversight to:

* Understand the capabilities and limitat...
Step 2: Observer Agent auditing answer...

    Grounding score: 5/5
    Source Relevance:  5/5
    Factual Integrity: 5/5
    Issues:            []
    VERDICT:           APPROVED

Query: What are the penalty tiers in the EU AI Act?
-------------------------------------------------------
Step 1: RAG Agent generating answer...
    Sources: ['EU AI Act 2024/1689, Article 99']
        Answer previ

,query,source,grounding_score,source_relevance,verdict
0,What does Article 14 require for human oversight?,"[EU AI Act 2024/1689, Article 14, EU AI Act 20...",5,5,APPROVED
1,What are the penalty tiers in the EU AI Act?,"[EU AI Act 2024/1689, Article 99]",5,5,APPROVED
2,What does NIS AI 600-1 say about hallucination?,"[NIST AI 600-1, Generative AI Profile]",5,5,APPROVED


## Findings: Observer Agent Architecture

**System:** gemini-flash-latest dual-agent pipeline
**Architecture:** RAG Agent + Observer Agent
**Queries tested:** 3
**Date:** June 2026
**Regulatory mapping:** NIST AI 600-1 output monitoring,
                        EU AI Act Article 14 human oversight

### Pipeline Results

| Query | Sources Retrieved | Grounding | Relevance | Integrity | Verdict |
|---|---|---|---|---|---|
| Article 14 human oversight | Article 14, Article 10 | 5/5 | 5/5 | 5/5 | APPROVED |
| Penalty tiers | Article 99 only | 5/5 | 5/5 | 5/5 | APPROVED |
| NIST AI 600-1 hallucination | NIST AI 600-1 only | 5/5 | 5/5 | 5/5 | APPROVED |

Overall: 3 out of 3 APPROVED (100%)

### Key Findings

1. The two-agent architecture produces fully audited
   responses with complete answers. Every RAG response
   includes the full answer grounded in source documents,
   not just source citations. The Observer Agent
   independently verified the complete answer content
   before assigning scores. Terminal output was truncated
   in earlier testing but CSV records confirm full
   answers including all penalty tiers, article
   requirements, and framework details were generated
   and audited correctly.

2. The Lesson 1 retrieval quality fix was confirmed.
   The penalty tier query now correctly retrieves
   Article 99 only, not Article 5. The domain-specific
   scoring boost added between Lesson 1 and Lesson 2
   resolved the incorrect retrieval finding documented
   earlier.

3. Robustness finding: the retriever correctly
   identified NIST AI 600-1 despite a typo in the
   query ("NIS AI 600-1" instead of "NIST AI 600-1").
   Partial keyword matching provides graceful handling
   of minor input errors.

4. Observer Agent produces structured JSON verdicts
   consistently. The three-tier APPROVED/FLAGGED/REJECTED
   system mirrors the CRITICAL/HIGH/PASS severity
   structure built in Phase 6 parameter validation.
   Consistent governance architecture across phases.

5. Answer completeness confirmed. The penalty tier
   query produced a complete answer listing all three
   Article 99 tiers including the 35 million euros or
   7% tier for prohibited practices, the 15 million
   euros or 3% tier for high-risk violations, and the
   7.5 million euros or 1% tier for incorrect
   information. The RAG pipeline is producing
   substantive regulatory guidance grounded in
   verified source documents, not surface-level
   citations.

### Architecture Significance

The Observer Agent implements EU AI Act Article 14
human oversight in automated form. Rather than a
human reviewing every RAG response manually, the
Observer Agent provides a first-pass automated audit
that flags problems for human review. This is the
hybrid human-AI oversight model that Article 14
envisions for high-risk AI deployments.

### NIST AI 600-1 Mapping
Output monitoring: Every RAG response independently
audited by Observer Agent before delivery.
Hallucination tracking: Grounding score measures
how well answers stay within source documents.
Source lineage: Every response traces back to
specific regulatory documents.

### Connection to Phase 7 Lesson 3
The Observer Agent currently audits AFTER generation.
Lesson 3 adds a semantic intent check that audits
BEFORE generation, intercepting suspicious queries
before the RAG pipeline even runs. Together they
form a complete pre-execution and post-execution
governance layer.

### Critical Limitation: Same-Family Observer Bias

Both the RAG Agent and Observer Agent use
gemini-flash-latest. This replicates the leniency
bias finding from Phase 3 where Gemini judged
Gemini outputs as perfect 5/5.

The Observer cannot detect blind spots it shares
with the RAG Agent. Perfect scores in this test
may reflect model-family agreement rather than
genuine quality verification.

Production mitigation: deploy a different model
family as the Observer Agent. The RAG Agent
generates the answer. An independent model audits
it. Cross-model auditing eliminates shared bias
and produces more objective quality verdicts.